In [ ]:
#!/usr/bin/env python
# coding: utf-8
"""
Community-detection check against known earthquake sequences — Italy (1985-2025)

Do the detected communities recover real, documented seismic sequences? We
isolate well-known Italian sequences (L'Aquila 2009, Amatrice-Norcia 2016,
Emilia 2012, Umbria-Marche 1997) by a space-time box around each mainshock,
then test whether community detection groups each sequence's events into a
single community instead of scattering them.
"""

import logging
from pathlib import Path

import networkx as nx
import pandas as pd
import plotly.io as pio
import seaborn as sns
from shapely.geometry import Point, Polygon

from src.network_custom import build_abe_suzuki_network_custom_hybrid
from src.network import discretize_space_3d
from src.community_custom import (
    run_louvain_directed_hybrid,
    run_infomap_hybrid,
    plot_community_geo_hybrid,
)
from src.known_sequences import (
    label_known_sequences,
    sequence_community_concentration,
    plot_sequence_community_geo,
)
from src.plotutils import setup_matplotlib, configure_saves

try:
    from IPython.display import display
except ImportError:
    display = print  # type: ignore[assignment]

logging.basicConfig(level=logging.INFO, format="%(levelname)s  %(message)s")
# Silence chatty third-party loggers (Kaleido/Chromium PDF export spam, matplotlib
# font cache, PIL image debug, asyncio cleanup) — keep our own INFO messages visible.
for _noisy in ("kaleido", "choreographer", "logistro", "matplotlib",
               "PIL", "urllib3", "asyncio"):
    logging.getLogger(_noisy).setLevel(logging.WARNING)
sns.set_theme(style="whitegrid")
pio.renderers.default = "notebook"

# ── Global configuration ─────────────────────────────────────────────────────
DATA_DIR    = Path("data/INGV")
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)
(RESULTS_DIR / "data").mkdir(exist_ok=True)
CUT_YEAR   = 1985
TARGET_CRS = "epsg:32632"             # UTM Zone 32N — Italy

# Days *before* each known mainshock to include in the sequence's space-time
# box (catches foreshocks). Per-sequence aftershock window length lives in the
# SEQUENCES list below (the ``days`` key). Increase PRE_DAYS to study earlier
# precursory activity around the mainshock.
PRE_DAYS = 2.0

# Hybrid network parameters — Optuna TPE-optimal values (see
# network_hybrid_optuna.py). The Optuna objective was NMI(directed Louvain,
# InfoMap); the values below give the highest method-stability NMI = 0.758
# on the hybrid 30-km Italy giant.
CELL_SIZE          = 45
HYBRID_ALPHA       = 0.5391
HYBRID_TAU_DAYS    = 0.8561
HYBRID_R0          = 20.331
HYBRID_SPATIAL_KM  = 300.0     # fixed during Optuna search
HYBRID_TIME_HOURS  = 24        # fixed during Optuna search

_IT_BOUNDS = dict(west=3, east=22, south=34, north=48)
MAP_WIDTH  = 770
MAP_HEIGHT = 700

setup_matplotlib()
configure_saves(True, True, RESULTS_DIR / "figures" / "italy" / "known_eq")

# Known Italian sequences, defined as space-time boxes around each mainshock.
# Mainshock times / boxes follow the conventions in ITALY_preanalysis.py.
SEQUENCES = [
    {"name": "Umbria-Marche 1997",
     "mainshock_time": pd.to_datetime("1997-09-26 00:33:00", utc=True),
     "days": 60, "lat_range": (42.7, 43.4), "lon_range": (12.5, 13.3)},
    {"name": "L'Aquila 2009",
     "mainshock_time": pd.to_datetime("2009-04-06 01:32:40.4", utc=True),
     "days": 50, "lat_range": (41.5, 43.0), "lon_range": (12.5, 14.0)},
    {"name": "Emilia 2012",
     "mainshock_time": pd.to_datetime("2012-05-20 02:03:52", utc=True),
     "days": 50, "lat_range": (44.6, 45.3), "lon_range": (10.7, 11.9)},
    {"name": "Amatrice-Norcia 2016",
     "mainshock_time": pd.to_datetime("2016-08-24 01:36:32", utc=True),
     "days": 120, "lat_range": (42.0, 43.5), "lon_range": (12.5, 14.0)},
]

DETECTION_METHODS = [
    ("Louvain", lambda G: run_louvain_directed_hybrid(G, resolution=1.0)),
    ("InfoMap", lambda G: run_infomap_hybrid(G, directed=True, seed=42)),
]

## The Idea

A community partition is only meaningful if its communities are physical.
We test that directly: take four well-documented Italian sequences, isolate
each by a **space-time box** around its mainshock (the same lat/lon/time
windows used for the Omori fits in ``ITALY_preanalysis.py``), and ask whether
community detection **recovers** the sequence — i.e. whether its events fall
into a single community.

For each sequence we report:
- **purity** — fraction of the sequence's (partitioned) events in its single
most-common community. Purity $\to 1$ means the sequence is one community.
- **normalised entropy** — spread of the sequence across communities
($0$ = one community, $1$ = evenly scattered).
- **seq_frac_of_community** — what fraction of that dominant community the
sequence makes up. High purity *and* high share = community $\approx$
sequence (clean recovery); high purity but low share = the sequence was
swallowed by a much larger flow basin.

Directed Louvain (structural) and InfoMap (flow-based) are both checked, so
we can see which detection method tracks real aftershock sequences better.

## Data Loading

Load the INGV catalog, restrict to $\text{year} \geq$ ``CUT_YEAR`` and crop
to the Italian polygon. The full magnitude-bearing catalog is reused so the
known-sequence boxes and the per-event community lookup share one event set.

In [ ]:
print("Loading Italy earthquake catalog...")
df = pd.read_csv(DATA_DIR / "italy_earthquakes_1985_2025.csv")
df["time"] = pd.to_datetime(df["time"], utc=True)
df_net = (
    df[df["time"].dt.year >= CUT_YEAR]
    .sort_values("time")
    .reset_index(drop=True)
)

_ITALY_POLYGON = Polygon([
    (13.5, 44.5), (19.0, 41.2), (19.0, 35.6), (11.7, 35.5), (11.6, 37.9),
    (7.2, 38.0), (7.2, 42.6), (5.2, 45.5), (11.6, 48.4), (16.0, 46.7), (13.5, 44.5),
])
df_net = df_net[df_net.apply(
    lambda r: _ITALY_POLYGON.contains(Point(r["longitude"], r["latitude"])), axis=1
)].reset_index(drop=True)
print(f"After polygon filter: {len(df_net):,} earthquakes")

## Isolate Known Sequences

Discretise the catalog at the network's cell size (so each event carries the
``cell_id`` that matches a network node), then label every event with the
known sequence whose space-time box contains it. The per-sequence counts
below confirm the boxes capture the expected aftershock bursts.

In [ ]:
df_grid = discretize_space_3d(df_net, CELL_SIZE, target_crs=TARGET_CRS, info=False)
df_grid = label_known_sequences(df_grid, SEQUENCES, pre_days=PRE_DAYS)

counts = df_grid["sequence"].value_counts(dropna=True)
print("Events isolated per known sequence (space-time box):")
display(counts.to_frame("n_events"))
print(f"Total labelled: {int(counts.sum()):,} of {len(df_grid):,} events")

## Network and Community Detection

Build the **hybrid** Abe-Suzuki network at Optuna's TPE-optimal parameters
(cell 45 km, $\alpha = 0.5391$, $r_0 = 20.331$ km, $\tau = 0.8561$ d,
with hard filters $\Delta r \leq 300$ km and $\Delta t \leq 24$ h on top
of the soft weights), and detect communities with directed Louvain and
InfoMap on its giant component. These parameters give the highest
method-stability NMI (Louvain $\leftrightarrow$ InfoMap = 0.758 on the
hybrid 30 km giant), meaning they produce the cleanest community
structure recoverable from this catalog.

**Reading the maps:** events are coloured by their *cell's* community, so a
community renders as the footprint of its cells — a community that is a
narrow column of cells (e.g. the N–S Apennine fault core) appears as a
grid-aligned rectangle. That blockiness is the discretisation, not a
spurious community; shrinking the cell size makes the blocks smaller but
cannot remove them.

In [ ]:
G = build_abe_suzuki_network_custom_hybrid(
    df_net, cell_size_km=CELL_SIZE,
    spatial_threshold_km=HYBRID_SPATIAL_KM,
    time_threshold_sec=HYBRID_TIME_HOURS * 3600,
    target_crs=TARGET_CRS,
    alpha=HYBRID_ALPHA, tau=HYBRID_TAU_DAYS * 86400.0, r0=HYBRID_R0,
    info=True,
)
G_giant = G.subgraph(max(nx.weakly_connected_components(G), key=len)).copy()
print(f"Hybrid network: {G.number_of_nodes():,} nodes / {G.number_of_edges():,} edges | "
      f"giant {G_giant.number_of_nodes():,}")

partitions = {}
for method_name, detect_fn in DETECTION_METHODS:
    cmap = detect_fn(G_giant)
    partitions[method_name] = cmap
    print(f"{method_name}: {len(set(cmap.values()))} communities")

## Recovery Check — Concentration Table

For each detection method, measure how cleanly every known sequence maps onto
the communities. ``purity`` near 1 with ``norm_entropy`` near 0 means the
sequence is recovered as essentially one community; ``seq_frac_of_community``
then tells whether that community is basically the sequence or a much bigger
basin that absorbed it.

In [ ]:
for method_name, cmap in partitions.items():
    comm_event_counts = (
        df_grid["cell_id"].map(cmap).dropna().astype(int).value_counts().to_dict()
    )
    conc = sequence_community_concentration(df_grid, cmap, comm_event_counts)
    print(f"\n=== {method_name} — sequence recovery ===")
    display(conc)
    conc.insert(0, "method", method_name)
    conc.insert(0, "network", "hybrid")
    conc.to_csv(
        RESULTS_DIR / "data" / f"known_eq_recovery_hybrid_{method_name.lower()}.csv",
        index=False,
    )

## Sequence Community Maps

Geographic view: each known sequence's events coloured by their detected
community (directed Louvain). A single colour per panel confirms the sequence
sits in one community; a colour mix means the partition split it. These make
the purity numbers above visually concrete.

In [ ]:
cmap_louvain = partitions["Louvain"]
for s in SEQUENCES:
    plot_sequence_community_geo(
        df_grid, cmap_louvain, s["name"],
        title=f"hybrid {CELL_SIZE} km (Optuna best)", method_name="Louvain",
        zoom=7, height=MAP_HEIGHT, width=MAP_WIDTH,
        save_name=f"seq_{s['name'].split()[0].lower()}_louvain_hybrid",
    )

## Context — Full Community Map

For reference, the full directed-Louvain community map of the network. The
recovered sequence communities above are sub-regions of this map; comparing
the two shows whether known sequences correspond to whole communities or to
dense cores within larger ones.

In [ ]:
plot_community_geo_hybrid(
    G_giant, cmap_louvain,
    title=f"Hybrid Italy ({CELL_SIZE} km, Optuna best)",
    center_lat=41.9, center_lon=12.5, zoom=0,
    height=MAP_HEIGHT, width=MAP_WIDTH, bounds=_IT_BOUNDS,
    min_community_size=20, method_name="Louvain (directed)",
)